# Análise Exploratória de Dados - Mercado Imobiliário de Fortaleza (ITBI)

**Contexto:** Após a extração, limpeza e georreferenciamento dos dados públicos de ITBI da Prefeitura de Fortaleza, esta etapa visa explorar os padrões ocultos nos dados. 

**Objetivos desta Análise:**
1. Compreender a distribuição do volume de vendas por tipo e padrão de imóvel.
2. Avaliar a dispersão do Valor do Metro Quadrado (M²) e tratar valores atípicos (*outliers*).
3. Identificar os bairros mais valorizados da cidade.
4. Mapear geograficamente as manchas de valorização imobiliária.

In [1]:
# Importando as bibliotecas de manipulação e visualização
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.io as pio
from pathlib import Path

# Configurações visuais globais
sns.set_theme(style="whitegrid")
pio.templates.default = "plotly_white"
pd.options.display.float_format = '{:,.2f}'.format # Tira números de notação científica

## 1. Carregamento dos Dados Refinados
Vamos importar o dataset que foi previamente limpo e filtrado (contendo apenas imóveis Residenciais e Comerciais).

In [2]:
# Carregar o dataset limpo que criamos no passo anterior
df = pd.read_excel( Path.cwd().parent / "dataset_filtered.xlsx")

print(f"Dataset carregado com {df.shape[0]} registros e {df.shape[1]} atributos.")
display(df[['bairro', 'tipo_uso_imovel', 'padrao_construcao', 'valor_m2']].head())

Dataset carregado com 85923 registros e 11 atributos.


,bairro,tipo_uso_imovel,padrao_construcao,valor_m2
0,JANGURUSSU,Residencial,Normal 2,"2,424.25"
1,RACHEL DE QUEIROZ,Residencial,Alto nivel 3,"3,969.87"
2,JARDIM CEARENSE,Residencial,Luxo 1,"4,355.06"
3,LAGOA REDONDA,Residencial,Alto nivel 1,"6,026.37"
4,GUARARAPES,Residencial,Luxo 2,"5,768.86"


In [3]:
# Selecionando apenas colunas numéricas e de área
colunas_numericas = ['area_edificada', 'vl_base_calculo', 'valor_m2']
df[colunas_numericas].describe().round(2)

,area_edificada,vl_base_calculo,valor_m2
count,"85,923.00","80,279.00","80,279.00"
mean,146.13,"465,897.43","3,263.19"
std,391.74,"1,158,498.55","2,291.28"
min,15.00,0.00,0.00
25%,66.95,"180,000.00","2,259.70"
50%,96.01,"280,000.00","3,000.00"
75%,152.24,"479,671.91","3,932.98"
max,"30,528.69","133,000,000.50","180,000.00"


## 2. Perfil das Transações Imobiliárias
Vamos analisar o volume de dados categóricos. A ideia é entender qual é o "feijão com arroz" do mercado cearense: o que mais se vende?

In [4]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# ==========================================
# Dados de proporção - Tipo de Uso
# ==========================================

prop_uso = (
    df['tipo_uso_imovel']
    .value_counts(normalize=True)
    .mul(100)
    .reset_index()
)

prop_uso.columns = ['tipo_uso_imovel', 'percentual']

# ==========================================
# Dados de proporção - Padrão Construção
# ==========================================

prop_padrao = (
    df['padrao_construcao']
    .value_counts(normalize=True)
    .mul(100)
    .reset_index()
)

prop_padrao.columns = ['padrao_construcao', 'percentual']

# ==========================================
# Criando subplots
# ==========================================

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=(
        "Proporção de Transações por Tipo de Uso",
        "Distribuição por Padrão de Construção"
    )
)

# ==========================================
# Gráfico 1
# ==========================================

fig.add_trace(
    go.Bar(
        x=prop_uso['percentual'],
        y=prop_uso['tipo_uso_imovel'],
        orientation='h',
        marker=dict(color=prop_uso['percentual'],
                    colorscale='agsunset_r'),
        
        text=[f"{v:.1f}%" for v in prop_uso['percentual']],
        textposition='outside',

        hovertemplate=
        "<b>%{y}</b><br>" +
        "Percentual: %{x:.2f}%<extra></extra>"
    ),
    row=1,
    col=1
)

# ==========================================
# Gráfico 2
# ==========================================

fig.add_trace(
    go.Bar(
        x=prop_padrao['padrao_construcao'],
        y=prop_padrao['percentual'],
        
        marker=dict(color=prop_padrao['percentual'],
                    colorscale='agsunset_r'),

        text=[f"{v:.1f}%" for v in prop_padrao['percentual']],
        textposition='outside',

        hovertemplate=
        "<b>%{x}</b><br>" +
        "Percentual: %{y:.2f}%<extra></extra>"
    ),
    row=1,
    col=2
)

# ==========================================
# Layout
# ==========================================

fig.update_layout(
    height=500,
    width=1200,
    showlegend=False,
    template="plotly_white",
    title="Distribuições das Transações Imobiliárias"
)

# Ajustes eixos
fig.update_xaxes(title_text="Percentual (%)", row=1, col=1)
fig.update_yaxes(title_text="", row=1, col=1)

fig.update_xaxes(title_text="Padrão", tickangle=45, row=1, col=2)
fig.update_yaxes(title_text="Percentual (%)", row=1, col=2)

fig.show()

## 3. Distribuição de Preços e Tratamento de Outliers
O mercado imobiliário é naturalmente enviesado: existem poucas propriedades de luxo extremo com valores que distorcem a média da cidade.

Para visualizar a distribuição real (o comportamento da grande massa do mercado), utilizaremos um **Boxplot** aplicando um corte no percentil 99. Isso remove o 1% dos imóveis mais caros apenas para fins de clareza visual nesta etapa.

In [5]:
# ==========================================
# Removendo extremos
# ==========================================

limite_superior = df['valor_m2'].quantile(0.99)

df_sem_extremos = df[
    df['valor_m2'] <= limite_superior
]

# ==========================================
# Formatação brasileira
# ==========================================

df_sem_extremos["valor_m2_fmt"] = (
    df_sem_extremos["valor_m2"]
    .apply(lambda x: f'R$ {x:,.2f}')
    .str.replace(",", "X", regex=False)
    .str.replace(".", ",", regex=False)
    .str.replace("X", ".", regex=False)
)

# ==========================================
# Boxplot interativo
# ==========================================

fig_box = px.box(
    df_sem_extremos,

    x='padrao_construcao',
    y='valor_m2',

    color='tipo_uso_imovel',

    points='outliers',
    
    custom_data=["valor_m2_fmt"],
    
    color_discrete_sequence=px.colors.sequential.Agsunset_r,

    title='Dispersão do Valor do M² por Padrão e Tipo<br><sup>Sem Outliers Extremos</sup>',

    labels={
        'padrao_construcao': 'Padrão de Construção',
        'valor_m2': 'Valor do m² (R$)',
        'tipo_uso_imovel': 'Tipo de Uso'
    },

    hover_data={
        'valor_m2_fmt': True,
        'valor_m2': False
    }
)

# ==========================================
# Layout
# ==========================================

fig_box.update_layout(
    template='plotly_white',

    xaxis_title='Padrão de Construção',
    yaxis_title='Valor do m² (R$)',

    legend_title='Tipo de Uso',

    height=600
)

# ==========================================
# Hover customizado
# ==========================================

fig_box.update_traces(
    hovertemplate=
    "<b>Padrão:</b> %{x}<br>" +
    "<b>Tipo:</b> %{fullData.name}<br>" +
    "<b>Valor do m²:</b> %{customdata[0]}<br>" +
    "<extra></extra>"
)

fig_box.show()

## 4. Análise Geográfica e de Valorização (Plotly)
Estes mesmos códigos servirão como base para os componentes do Dashboard no `Dash`.

### 4.1. Ranking: Top 10 Bairros com M² Mais Caro

In [6]:
# Agrupar por bairro
df_bairros = (
    df.groupby('bairro')['valor_m2']
      .mean()
      .reset_index()
)

# Top 10
df_top10 = (
    df_bairros
    .sort_values(by='valor_m2', ascending=False)
    .head(10)
)

# Valor numérico arredondado
df_top10['valor_m2_round'] = df_top10['valor_m2'].round(2)

# ==========================================
# Formatação brasileira
# ==========================================

df_top10['valor_m2_br'] = (
    df_top10['valor_m2_round']
    .apply(lambda x: f'R$ {x:,.2f}')
    .str.replace(',', 'X', regex=False)
    .str.replace('.', ',', regex=False)
    .str.replace('X', '.', regex=False)
)

# ==========================================
# Gráfico
# ==========================================

fig_bar = px.bar(
    df_top10,
    x='valor_m2_round',
    y='bairro',
    orientation='h',

    text='valor_m2_br',

    title='Top 10 Bairros Mais Valorizados (Média do M²)',

    labels={
        'valor_m2_round': 'Valor Médio (R$)',
        'bairro': 'Bairro'
    },

    color='valor_m2_round',
    color_continuous_scale='agsunset_r'
)

# ==========================================
# Layout
# ==========================================

fig_bar.update_traces(
    textposition='inside',

    hovertemplate=
    '<b>%{y}</b><br>' +
    'Valor Médio: %{text}<extra></extra>'
)

fig_bar.update_layout(
    yaxis={'categoryorder': 'total ascending'},
    coloraxis_showscale=False
)

fig_bar.show()

### 4.2. Mapa de Dispersão de Valores (Heatmap de Preços)
Visualização espacial de todas as transações. A cor representa o preço do metro quadrado e o tamanho da bolha reflete a área do imóvel.

*Nota: Dependendo do poder de processamento da máquina, plotar os 90 mil pontos simultaneamente pode causar lentidão no navegador.*

In [11]:
# ==========================================
# Limpeza
# ==========================================

df_mapa = df.dropna(
    subset=["latitude", "longitude", "area_edificada", "valor_m2"]
)

# Remover valores inválidos
df_mapa = df_mapa[
    (df_mapa["valor_m2"] > 0) &
    (df_mapa["area_edificada"] > 0)
]

# ==========================================
# Amostragem
# ==========================================

if len(df_mapa) > 10000:
    df_mapa = df_mapa.sample(n=10000, random_state=42)

# ==========================================
# Formatação brasileira
# ==========================================

df_mapa["valor_m2_fmt"] = (
    df_mapa["valor_m2"]
    .apply(lambda x: f'R$ {x:,.2f}')
    .str.replace(",", "X", regex=False)
    .str.replace(".", ",", regex=False)
    .str.replace("X", ".", regex=False)
)

df_mapa["vl_base_calculo_fmt"] = (
    df_mapa["vl_base_calculo"]
    .apply(lambda x: f'R$ {x:,.2f}')
    .str.replace(",", "X", regex=False)
    .str.replace(".", ",", regex=False)
    .str.replace("X", ".", regex=False)
)

df_mapa["area_edificada_fmt"] = (
    df_mapa["area_edificada"]
    .apply(lambda x: f'{x:,.1f}')
    .str.replace(",", "X", regex=False)
    .str.replace(".", ",", regex=False)
    .str.replace("X", ".", regex=False)
)

# ==========================================
# Mapa
# ==========================================

fig_map = px.scatter_mapbox(
    df_mapa,
    
    lat="latitude",
    lon="longitude",

    color="valor_m2",
    size="area_edificada",
    
    labels={"valor_m2": "Valor do m²"},

    hover_name="bairro",

    hover_data={
        "latitude": False,
        "longitude": False,

        "tipo_uso_imovel": True,

        "vl_base_calculo_fmt": True,
        "valor_m2_fmt": True,

        "valor_m2": False,
        "vl_base_calculo": False,

        "area_edificada_fmt": True
    },

    color_continuous_scale=px.colors.sequential.Magma_r,

    size_max=15,

    zoom=10.5,

    mapbox_style="carto-positron",

    title="Mapa de Transações: Valor do M² por Região em reais (R$)"
)

# ==========================================
# Tooltip customizado
# ==========================================

fig_map.update_traces(
    hovertemplate=
    "<b>%{hovertext}</b><br><br>" +

    "Tipo de uso: %{customdata[2]}<br>" +
    "Valor venal: %{customdata[3]}<br>" +
    "Valor do m²: %{customdata[4]}<br>" +
    "Área edificada: %{customdata[7]} m²" +

    "<extra></extra>"
)

# ==========================================
# Layout
# ==========================================

fig_map.update_layout(
    margin={"r":0, "t":50, "l":0, "b":0}
)

fig_map.show()

C:\Users\dougl\AppData\Local\Temp\ipykernel_13468\1331025151.py:54: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig_map = px.scatter_mapbox(
